In [1]:
import pandas, csv, mygene

In [2]:
# 1. table of expression for all conditions

results_folder = '/Users/adrian/research/bmcbf/025_isafjordur/results/000_quantification/'
results_folder = '/Users/adrian/Library/CloudStorage/ProtonDrive-adrian.rvk@proton.me-folder/scratch/'
expression_file_skmel = results_folder + 'DESeq2_TPM_values.skmel.tsv'
expression_file_u2o2 = results_folder + 'DESeq2_TPM_values.u2.tsv'

expression_skmel = pandas.read_csv(expression_file_skmel, sep='\t', index_col=0)
expression_u2os = pandas.read_csv(expression_file_u2o2, sep='\t', index_col=0)

print(expression_skmel.shape, expression_u2os.shape)

full_expression = pandas.concat([expression_skmel, expression_u2os], axis=1)
print(full_expression.shape)

selected_columns = ['SkMel28-MITFKO_ev_skmel28_rep1', 'SkMel28-MITFKO_ev_skmel28_rep2','SkMel28-MITFKO_ev_skmel28_rep3', 'SkMel28-MITFKO_ev_skmel28_rep4',
       'SkMel28-MITFKO_mitf_x6_rep1', 'SkMel28-MITFKO_mitf_x6_rep3','SkMel28-MITFKO_mitf_x6_rep4', 
        'U2-1_Sc_UT', 'U2-2_Sc_UT', 'U2-3_Sc_UT',
        'U2-1_siMi_UT', 'U2-2_siMi_UT', 'U2-3_siMi_UT']

expression = full_expression.loc[:, selected_columns]
expression

(39546, 7) (39546, 18)
(39546, 25)


,SkMel28-MITFKO_ev_skmel28_rep1,SkMel28-MITFKO_ev_skmel28_rep2,SkMel28-MITFKO_ev_skmel28_rep3,SkMel28-MITFKO_ev_skmel28_rep4,SkMel28-MITFKO_mitf_x6_rep1,SkMel28-MITFKO_mitf_x6_rep3,SkMel28-MITFKO_mitf_x6_rep4,U2-1_Sc_UT,U2-2_Sc_UT,U2-3_Sc_UT,U2-1_siMi_UT,U2-2_siMi_UT,U2-3_siMi_UT
ENSG00000000003,9.097209,9.806654,12.244351,12.179835,17.984601,12.499056,10.895395,30.818057,5.939035,28.165731,6.893577,40.127345,36.374311
ENSG00000000005,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.043649,0.000000,0.020145,0.024416
ENSG00000000419,152.065078,146.963798,159.427516,180.241348,197.693333,144.099788,136.583522,112.914933,75.822065,107.421701,68.466391,87.365983,84.883315
ENSG00000000457,4.644776,3.992320,6.102643,3.590144,5.672788,4.962348,4.242799,2.598995,10.594728,2.507932,10.456216,4.589684,4.514144
ENSG00000000460,20.780175,22.200583,30.912937,20.707054,26.297054,17.887952,14.196562,32.056208,23.911539,28.125365,16.831813,11.772422,11.618076
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ENSG00000291297,5.742134,5.220425,6.054220,5.305273,2.120733,2.756115,2.122733,4.408266,0.545854,4.523390,0.229724,2.258105,4.005492
ENSG00000291298,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.944026,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ENSG00000291299,1.784678,1.767580,1.662962,2.593021,0.870331,2.213504,1.361509,1.743303,2.385269,2.397874,3.476740,2.271456,2.192518
ENSG00000291300,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.455893,0.230373,0.362591,0.206109,0.314552,0.284173


In [3]:
# 2. the genes for the cell cycle and apoptosis categories
# the file was retrieved from enrichr as reactome pathways 2024

rows = []
with open('Reactome_Pathways_2024.txt', newline='') as f:
    reader = csv.reader(f, delimiter='\t')   # adjust delimiter
    for row in reader:
        rows.append(row)

df = pandas.DataFrame(rows)
mask = df[0].str.contains('TP53 Regulates', case=False, na=False)
new = df[mask]
new = new.set_index(0)
new

values = new.loc['TP53 Regulates Transcription of Cell Cycle Genes', :].dropna().to_list()
genes_cycle = [value for value in values if value != '']
print(len(genes_cycle), genes_cycle[:5])

values = new.loc['TP53 Regulates Transcription of Cell Death Genes', :].dropna().to_list()
genes_death = [value for value in values if value != '']
print(len(genes_death), genes_death[:5])

values = new.loc['TP53 Regulates Transcription of DNA Repair Genes', :].dropna().to_list()
genes_repair = [value for value in values if value != '']
print(len(genes_repair), genes_repair[:5])

49 ['ARID3A', 'CNOT11', 'CNOT10', 'CDC25C', 'CNOT6']
44 ['ZNF420', 'CREBBP', 'PRELID1', 'APAF1', 'BNIP3L']
62 ['JUN', 'SUPT16H', 'GTF2F1', 'MSH2', 'GTF2F2']


In [4]:
# 3. convert gene symbols into ensembls

mg = mygene.MyGeneInfo()

out = mg.querymany(genes_repair, scopes='symbol', fields='ensembl.gene', species='human', as_dataframe=True)
out = out.explode('ensembl.gene')
ids = out['ensembl.gene'].dropna().unique().tolist()
print(len(ids), ids[:5])




Input sequence provided is already in string format. No operation performed
Input sequence provided is already in string format. No operation performed


54 ['ENSG00000177606', 'ENSG00000092201', 'ENSG00000125651', 'ENSG00000095002', 'ENSG00000188342']


In [5]:
nan_rows = out[out['ensembl.gene'].isna()]
nan_rows


,_id,_score,ensembl.gene,ensembl
query,,,,
CDK7,1022,17.027145,NaN,"[{'gene': 'ENSG00000277273'}, {'gene': 'ENSG00..."
GTF2H1,2965,17.556274,NaN,"[{'gene': 'ENSG00000288114'}, {'gene': 'ENSG00..."
GTF2H2,2966,18.304321,NaN,"[{'gene': 'ENSG00000276910'}, {'gene': 'ENSG00..."
GTF2H4,2968,17.830267,NaN,"[{'gene': 'ENSG00000226384'}, {'gene': 'ENSG00..."
MDC1,9656,17.913400,NaN,"[{'gene': 'ENSG00000234012'}, {'gene': 'ENSG00..."
CTDP1,9150,17.628109,NaN,"[{'gene': 'ENSG00000060069'}, {'gene': 'ENSG00..."
POLR2A,5430,17.732336,NaN,"[{'gene': 'ENSG00000181222'}, {'gene': 'ENSG00..."
NELFE,7936,17.879396,NaN,"[{'gene': 'ENSG00000206357'}, {'gene': 'ENSG00..."


In [6]:
out.head(50)



,_id,_score,ensembl.gene,ensembl
query,,,,
JUN,3725,17.568882,ENSG00000177606,NaN
SUPT16H,11198,17.796005,ENSG00000092201,NaN
GTF2F1,2962,17.839700,ENSG00000125651,NaN
MSH2,4436,17.367363,ENSG00000095002,NaN
GTF2F2,2963,18.219427,ENSG00000188342,NaN
CDK9,1025,17.055676,ENSG00000136807,NaN
CDK7,1022,17.027145,NaN,"[{'gene': 'ENSG00000277273'}, {'gene': 'ENSG00..."
RAD51D,5892,17.145338,ENSG00000185379,NaN
FANCD2,2177,17.074186,ENSG00000144554,NaN


In [7]:
# go directly to do the heatmap

In [8]:
mask = df[0].str.contains('TP53 Regulates', case=False, na=False)
df[mask]


,0,1,2,3,4,5,6,7,8,9,...,2606,2607,2608,2609,2610,2611,2612,2613,2614,2615
1860,TP53 Regulates Metabolic Genes,,G6PD,GPX2,TIGAR,GSR,TXNRD1,TSC2,TSC1,PRKAB1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1861,TP53 Regulates Transcription of Caspase Activa...,,TP73,CASP6,APAF1,CASP10,CRADD,CASP2,ATM,NLRC4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1862,TP53 Regulates Transcription of Cell Cycle Genes,,ARID3A,CNOT11,CNOT10,CDC25C,CNOT6,CNOT7,RBL1,CNOT8,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1863,TP53 Regulates Transcription of Cell Death Genes,,ZNF420,CREBBP,PRELID1,APAF1,BNIP3L,CRADD,IGFBP3,TNFRSF10D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1864,TP53 Regulates Transcription of DNA Repair Genes,,JUN,SUPT16H,GTF2F1,MSH2,GTF2F2,CDK9,CDK7,RAD51D,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1865,TP53 Regulates Transcription of Death Receptor...,,TP73,PPP1R13B,FAS,IGFBP3,TMEM219,TNFRSF10D,TNFRSF10C,TNFRSF10B,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1866,TP53 Regulates Transcription of Genes Involved...,,ZNF420,CREBBP,PPP1R13B,PRELID1,BNIP3L,TP53INP1,TRIAP1,BBC3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1867,TP53 Regulates Transcription of Genes Involved...,,CDKN1B,CDKN1A,ARID3A,CCNA2,CCNA1,PCBP4,CCNE2,CCNE1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1868,TP53 Regulates Transcription of Genes Involved...,,PCNA,GADD45A,AURKA,PRMT1,CDC25C,RBL1,TFDP1,CCNB1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1869,TP53 Regulates Transcription of Additional Cel...,,CNOT6L,NPM1,BTG2,PLK3,PLK2,CNOT11,CNOT10,CDC25C,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
